# 04 — In-domain generation with EmotionPlex-7b (merged, bf16)

Generates NEW agent replies with the emotion-tuned model so the head can be
trained on Mimi/VARM3-domain audio (the dataset's Higgs voices are off-domain).

- Feeds the dataset **user channel** (right), then silence; the model free-runs
  exactly like the live duplex path (`moshi_tokens=None` → sampled).
- Text prompt `"Answer with a {emotion} tone."` + **your VARM3.pt** voice prompt
  (this is the generation step VARM3 is for).
- Stop rule: model fell silent for `STOP_SILENCE_S` after having spoken; hard cap
  `MAX_REPLY_S`. QA rejects too-short / silent / over-talkative outputs.
- ~3–6 s per sample on a 5090 → 4k samples ≈ 4–7 h. Resume-safe.

**Restart the kernel before this notebook** if you ran 02/03 in this session
(frees the bnb4 weights; the bf16 model needs ~17 GB VRAM).


In [ ]:
from pathlib import Path
import os, sys, json, time
import numpy as np
import torch

sys.path.insert(0, os.getcwd())
import eh_common as eh

P = eh.get_paths()   # override: P = eh.get_paths("/your/root") or set EMOTION_HEAD_ROOT
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16
print("ROOT:", P.ROOT, "| device:", DEVICE)
for k in ("MOSHI_BNB4", "MIMI_FILE", "SPM_FILE", "VOICE_PT"):
    p = Path(getattr(P, k))
    print(f"  {k}: {'OK' if p.exists() else 'MISSING'}  {p}")


In [ ]:
import collections
N_SAMPLES = 4000
MAX_REPLY_S = 18.0
STOP_SILENCE_S = 2.0
MIN_REPLY_S = 0.5


In [ ]:
import pandas as pd
df = pd.read_parquet(P.DATA / "manifest.parquet")
df = df[(df.has_wav) & (df.split == "train")].copy()
src = df.sample(min(len(df), N_SAMPLES), random_state=eh.SEED).reset_index(drop=True)
samples = [dict(sample_id=f"gen_{i:05d}__{r.clip_id}", clip_id=r.clip_id,
                wav_path=r.wav_path, emotion=eh.EMOTIONS[i % 4], user_id=r.user_id)
           for i, r in enumerate(src.itertuples())]
print(f"{len(samples)} generation targets | instructed emotions:",
      collections.Counter(s["emotion"] for s in samples))


In [ ]:
from huggingface_hub import snapshot_download
GEN_DIR = P.ROOT / "checkpoints" / "emotionplex-7b"
if not (GEN_DIR / "model.safetensors").exists():
    snapshot_download(eh.EMOPLEX_REPO, local_dir=str(GEN_DIR),
                      allow_patterns=["*.safetensors", "*.json", "*.txt", "*.model"])
print(sorted(p.name for p in GEN_DIR.iterdir()))
eh.free_gpu()
print(f"free vram: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1e9:.1f} GiB")
sp = eh.load_sp(P)
mimi = eh.load_mimi(P, DEVICE)
lm = eh.load_lm_bf16(GEN_DIR / "model.safetensors", DEVICE, DTYPE)
print(f"cuda mem: {torch.cuda.memory_allocated()/1e9:.1f} GiB")
lm_gen = eh.make_lm_gen(lm, DEVICE)
eh.install_hidden_step(lm_gen)


In [ ]:
assert P.VOICE_PT.exists(), "VARM3.pt is required for generation (your voice prompt)"
base_state = eh.snapshot_streaming_state(lm_gen)
prompt_states = {}
for e in eh.EMOTIONS:
    eh.restore_streaming_state(lm_gen, mimi, base_state)
    prompt_states[e] = eh.apply_prompts_and_snapshot(
        lm_gen, mimi, sp, str(P.VOICE_PT), f"Answer with a {e} tone.")
    print("prompt state ready:", e)


In [ ]:
from tqdm.auto import tqdm
N_GPUS = torch.cuda.device_count()
previews = {}
if N_GPUS > 1:
    # one spawned worker per GPU (bf16 emotionplex-7b on each), per-GPU shard
    # prefix (gen_g<i>) and log (generate_g<i>.log). Resume: re-run this cell.
    print(f"multi-GPU: {N_GPUS} GPUs -> {N_GPUS} generation workers (round-robin)")
    for _n in ("lm", "lm_gen", "mimi"):
        globals().pop(_n, None)     # free the in-kernel model so GPU0 has room
    eh.free_gpu()
    eh.generate_multi(samples, str(P.ROOT), N_GPUS,
                      dict(model_st=str(GEN_DIR / "model.safetensors"),
                           max_reply_s=MAX_REPLY_S, min_reply_s=MIN_REPLY_S,
                           stop_silence_s=STOP_SILENCE_S))
    print("all workers done | shards:", len(list(P.GEN.glob("gen*_shard_*.pt"))),
          "| logs:", sorted(p.name for p in P.GEN.glob("generate*.log")))
else:
    log = eh.ProgressLogger(P.GEN / "generate.log", echo_every=25)
    writer = eh.ShardWriter(P.GEN, "gen", clips_per_shard=128)
    qa_fail = collections.Counter()
    todo = [s for s in samples if s["sample_id"] not in writer.completed]
    total_target = len(writer.completed) + len(todo)
    log.log(f"=== generation START | completed={len(writer.completed)} todo={len(todo)} "
            f"target={total_target} | max_reply={MAX_REPLY_S}s stop_silence={STOP_SILENCE_S}s",
            force=True)
    t0 = time.time()
    failed, times = 0, []
    min_frames = int(MIN_REPLY_S * eh.FRAME_HZ)
    for s in tqdm(todo, total=len(todo)):
        ct0 = time.time()
        try:
            user = eh.read_channel(s["wav_path"], 1)   # right channel = user
            eh.restore_streaming_state(lm_gen, mimi, prompt_states[s["emotion"]])
            out = eh.generate_reply(lm_gen, mimi, user, max_reply_s=MAX_REPLY_S,
                                    min_reply_s=MIN_REPLY_S, stop_silence_s=STOP_SILENCE_S,
                                    device=DEVICE)
            T = out["codes"].shape[0]
            reply_frames = T - out["n_user_frames"]
            speak = float((out["rms"][out["n_user_frames"]:] > 0.006).mean()) if reply_frames > 0 else 0.0
            if reply_frames < min_frames:
                qa_fail["too_short"] += 1
                continue
            if not (0.05 <= speak <= 0.97):
                qa_fail["speak_fraction"] += 1
                continue
            rec = dict(clip_id=s["sample_id"], src_clip=s["clip_id"], user_id=s["user_id"],
                       emotion=s["emotion"], label=eh.EMO2ID[s["emotion"]],
                       codes=out["codes"],
                       text_tokens=torch.from_numpy(out["text_tokens"]).to(torch.int32),
                       pcm=torch.from_numpy(out["pcm"]).to(torch.float16),
                       rms=torch.from_numpy(out["rms"]).to(torch.float16),
                       n_user_frames=out["n_user_frames"], user_codes=out["user_codes"])
            writer.add(rec)
            previews.setdefault(s["emotion"], rec)
        except Exception as e:
            failed += 1
            log.log(f"ERROR sample={s['sample_id']} ({s['emotion']}): "
                    f"{type(e).__name__}: {e} | skipped (resume-safe)", force=True)
            continue
        dt = time.time() - ct0
        times.append(dt)
        completed = len(writer.completed)
        remaining = max(0, total_target - completed)
        avg = sum(times) / len(times)
        log.log(f"sample {completed}/{total_target} | +{dt:.1f}s avg={avg:.1f}s "
                f"| left={remaining} ETA={avg * remaining / 3600:.2f}h "
                f"| qa_rej={sum(qa_fail.values())} failed={failed} | {s['sample_id']} ({s['emotion']})")
    writer.close()
    log.log(f"=== generation END | completed={len(writer.completed)}/{total_target} "
            f"QA rejected={dict(qa_fail)} failed={failed} "
            f"| elapsed={(time.time() - t0) / 3600:.2f}h", force=True)
    print("generated:", len(writer.completed), "| QA rejected:", dict(qa_fail),
          "| failed:", failed, "| log:", log.path)


In [ ]:
from IPython.display import Audio, display
if not previews:    # multi-GPU run (or resumed kernel): pull previews from shards
    for rec in eh.iter_shard_records(P.GEN, "gen"):
        if rec["emotion"] not in previews:
            previews[rec["emotion"]] = rec
        if len(previews) == len(eh.EMOTIONS):
            break
for e in eh.EMOTIONS:
    rec = previews.get(e)
    if rec is None:
        print(e, ": no preview")
        continue
    dur = rec["pcm"].shape[0] / eh.SAMPLE_RATE
    print(f"{e}: {rec['clip_id']} ({dur:.1f}s)")
    display(Audio(rec["pcm"].float().numpy(), rate=eh.SAMPLE_RATE))
for _n in ("lm", "lm_gen", "mimi"):
    globals().pop(_n, None)
eh.free_gpu()
print("→ continue with 05_extract_indomain_and_train_phaseB.ipynb")
